# LocaleLive — LangSmith Trace Explorer

Uses the official **LangSmith SDK** (`langsmith.Client`) to pull LangGraph execution traces.

**What you get**
- All root runs (one per query): latency, token usage, status
- Per-agent node breakdown: timing, tokens
- Routing path reconstruction per trace
- Full trace replay with every node's input/output
- LLM node drill-down: exact messages + reply

**Requires:** `langsmith>=0.1`, `pandas`, `matplotlib`, `seaborn`, `python-dotenv`

In [1]:
%pip install "langsmith>=0.4" pandas matplotlib seaborn python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%matplotlib inline
import os
import json
from datetime import datetime, timezone, timedelta
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from dotenv import load_dotenv
from langsmith import Client

load_dotenv(Path("credentials.env"))

LANGCHAIN_API_KEY = os.environ["LANGCHAIN_API_KEY"]

client = Client(api_key=LANGCHAIN_API_KEY)

import langsmith as _ls
print(f"langsmith version : {_ls.__version__}")
print("Connected OK")

## 1 — List projects

In [3]:
projects = list(client.list_projects())

proj_df = pd.DataFrame([
    {
        "name":       p.name,
        "id":         str(p.id),
        "created_at": p.start_time,
    }
    for p in projects
])
print(f"{len(proj_df)} project(s)")
proj_df

5 project(s)


,name,id,created_at
0,IoT-ASE,fbff77bc-fa18-4a96-a1c5-41434c9c2559,2024-10-28 18:19:02.316770+00:00
1,Real-Time IoT Engine,7a00fa21-295d-45cd-89f3-80fcc763ee37,2024-10-21 14:55:18.915453+00:00
2,LangGraph IoT Engine,1ec8f97c-4ef6-4e71-b4ee-8598ebcb7cc7,2024-08-11 04:34:07.644514+00:00
3,LangGraph Tutorial,69c2faf7-e354-4684-ac74-730d280d99e6,2024-08-08 18:35:42.783122+00:00
4,default,42295fe6-e15d-48ae-85cc-1a901e8faee3,2024-08-05 22:29:20.973000+00:00


In [4]:
# Set PROJECT_NAME to any name from the table above.
# LocaleLive Lambda writes to 'default' unless LANGCHAIN_PROJECT env var was set.
PROJECT_NAME = "default"

LOOKBACK_HOURS = 72
start_dt = datetime.now(timezone.utc) - timedelta(hours=LOOKBACK_HOURS)

print(f"Project : {PROJECT_NAME}")
print(f"Window  : last {LOOKBACK_HOURS}h (from {start_dt.strftime('%Y-%m-%d %H:%M UTC')})")

Project : default
Window  : last 72h (from 2026-07-09 21:20 UTC)


## 2 — Pull all root runs (one per user query)

LangGraph records one root run named `LangGraph` per `PUT /query` call.

In [5]:
import itertools

# LangSmith API max per-page is 100; list_runs is a lazy iterator that
# paginates automatically — do NOT pass limit= directly to the call.
# Use itertools.islice as a safety cap on total rows fetched.
MAX_RUNS = 2000

raw_runs = list(itertools.islice(
    client.list_runs(
        project_name=PROJECT_NAME,
        start_time=start_dt,
        is_root=True,
    ),
    MAX_RUNS,
))
print(f"Fetched {len(raw_runs)} root runs")

records = []
for r in raw_runs:
    dur = (
        (r.end_time - r.start_time).total_seconds()
        if r.end_time and r.start_time else None
    )
    records.append({
        "run_id":            str(r.id),
        "trace_id":          str(r.trace_id) if r.trace_id else str(r.id),
        "name":              r.name,
        "status":            r.status,
        "start_time":        r.start_time,
        "end_time":          r.end_time,
        "duration_s":        dur,
        "total_tokens":      r.total_tokens or 0,
        "prompt_tokens":     r.prompt_tokens or 0,
        "completion_tokens": r.completion_tokens or 0,
        "total_cost":        float(r.total_cost or 0),
        "error":             r.error,
    })

runs_df = pd.DataFrame(records).sort_values("start_time").reset_index(drop=True)
print(runs_df["status"].value_counts().to_string())
if runs_df["duration_s"].notna().sum() > 0:
    d = runs_df["duration_s"].dropna()
    print(f"\nLatency p50={d.quantile(0.5):.2f}s  p95={d.quantile(0.95):.2f}s  max={d.max():.2f}s")
runs_df.tail(5)

Fetched 12 root runs
status
success    12

Latency p50=2.61s  p95=3.09s  max=3.11s


,run_id,trace_id,name,status,start_time,end_time,duration_s,total_tokens,prompt_tokens,completion_tokens,total_cost,error
7,4cc206fe-5237-4acf-afe7-ce9b34f26d05,4cc206fe-5237-4acf-afe7-ce9b34f26d05,LangGraph,success,2026-07-12 20:36:43.518830+00:00,2026-07-12 20:36:46.595712+00:00,3.076882,2209,1181,1028,0.0,None
8,e8dfe8f1-00da-4ae5-82dd-d34f765c9a0b,e8dfe8f1-00da-4ae5-82dd-d34f765c9a0b,LangGraph,success,2026-07-12 20:37:31.479588+00:00,2026-07-12 20:37:33.865620+00:00,2.386032,2294,1568,726,0.0,None
9,4af3d0b2-7346-4c97-945e-2214253e59b6,4af3d0b2-7346-4c97-945e-2214253e59b6,LangGraph,success,2026-07-12 20:38:31.205183+00:00,2026-07-12 20:38:34.115598+00:00,2.910415,1818,1184,634,0.0,None
10,6f802895-f8d7-47b2-aeff-92db4d36ab99,6f802895-f8d7-47b2-aeff-92db4d36ab99,LangGraph,success,2026-07-12 20:45:23.206107+00:00,2026-07-12 20:45:26.318343+00:00,3.112236,1817,1243,574,0.0,None
11,2a1c5b2a-944f-4e3a-b6b3-69d14d6ca0d9,2a1c5b2a-944f-4e3a-b6b3-69d14d6ca0d9,LangGraph,success,2026-07-12 20:45:49.489292+00:00,2026-07-12 20:45:51.787260+00:00,2.297968,1850,1176,674,0.0,None


## 3 — Latency & token usage overview

In [ ]:
if not runs_df.empty and runs_df["duration_s"].notna().sum() >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    d = runs_df["duration_s"].dropna()
    axes[0].hist(d, bins=max(5, min(30, len(d))), color="steelblue", edgecolor="white")
    for pct, color, lbl in [(0.5, "lime", "p50"), (0.95, "orange", "p95")]:
        v = d.quantile(pct)
        axes[0].axvline(v, linestyle="--", color=color, linewidth=1.5, label=f"{lbl}={v:.1f}s")
    axes[0].set_title("End-to-end latency")
    axes[0].set_xlabel("Duration (s)")
    axes[0].set_ylabel("Count")
    axes[0].legend(fontsize=8)

    t = runs_df["total_tokens"]
    if t.sum() > 0:
        axes[1].hist(t, bins=max(5, min(30, len(t))), color="seagreen", edgecolor="white")
        axes[1].set_title("Total tokens per query")
        axes[1].set_xlabel("Tokens")
        axes[1].set_ylabel("Count")
    else:
        axes[1].text(0.5, 0.5, "No token data\n(LLM calls not traced)",
                     ha="center", va="center", transform=axes[1].transAxes)
        axes[1].set_title("Total tokens per query")

    status_counts = runs_df["status"].value_counts()
    axes[2].pie(status_counts, labels=status_counts.index, autopct="%1.0f%%",
                colors=["steelblue", "crimson", "orange"][:len(status_counts)])
    axes[2].set_title("Run status")

    plt.tight_layout()
    plt.show()
    plt.close("all")

## 4 — Pull all child runs (individual LangGraph nodes)

In [7]:
# No limit= param — iterator paginates automatically; islice caps total rows
all_runs_raw = list(itertools.islice(
    client.list_runs(
        project_name=PROJECT_NAME,
        start_time=start_dt,
    ),
    10_000,
))
print(f"Total runs fetched (all depths): {len(all_runs_raw)}")

node_records = []
for r in all_runs_raw:
    dur = (
        (r.end_time - r.start_time).total_seconds()
        if r.end_time and r.start_time else None
    )
    node_records.append({
        "run_id":            str(r.id),
        "trace_id":          str(r.trace_id) if r.trace_id else None,
        "parent_run_id":     str(r.parent_run_id) if r.parent_run_id else None,
        "name":              r.name,
        "run_type":          r.run_type,
        "status":            r.status,
        "start_time":        r.start_time,
        "duration_s":        dur,
        "total_tokens":      r.total_tokens or 0,
        "prompt_tokens":     r.prompt_tokens or 0,
        "completion_tokens": r.completion_tokens or 0,
        "error":             r.error,
    })

nodes_df = pd.DataFrame(node_records).sort_values("start_time").reset_index(drop=True)
print("\nRun types:")
print(nodes_df["run_type"].value_counts().to_string())
print("\nTop 15 node names:")
print(nodes_df["name"].value_counts().head(15).to_string())

Total runs fetched (all depths): 204

Run types:
run_type
chain    180
llm       24

Top 15 node names:
name
ChatGroq                                                                                                                            24
LangGraph                                                                                                                           12
__start__                                                                                                                           12
assistant_agent                                                                                                                     12
ChannelWrite<assistant_agent,messages,query,context,response,call,node,location_finder_results,thread_summary,places,search_type    12
assitant_router                                                                                                                     12
GoogleMaps                                                                       

## 5 — Per-agent latency and token breakdown

In [ ]:
AGENT_NODES = [
    "assistant_agent", "IoT_engine", "GoogleMaps",
    "scrapper", "generator_agent", "reviewer_agent", "finalize_turn"
]

agent_df = nodes_df[nodes_df["name"].isin(AGENT_NODES)].copy()
print(f"{len(agent_df)} agent-node runs")

if not agent_df.empty:
    stats = (
        agent_df.groupby("name")
        .agg(
            count=("run_id", "count"),
            p50_s=("duration_s", lambda x: round(x.quantile(0.5), 3)),
            p95_s=("duration_s", lambda x: round(x.quantile(0.95), 3)),
            avg_tokens=("total_tokens", lambda x: round(x.mean(), 1)),
            total_tokens=("total_tokens", "sum"),
            errors=("error", lambda x: x.notna().sum()),
        )
        .reindex([n for n in AGENT_NODES if n in agent_df["name"].values])
    )
    display(stats)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    order = agent_df.groupby("name")["duration_s"].median().sort_values().index.tolist()
    sns.boxplot(data=agent_df, x="name", y="duration_s", order=order, ax=axes[0])
    axes[0].set_title("Latency per agent node")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Duration (s)")
    axes[0].tick_params(axis="x", rotation=20)

    tok_vals = stats["avg_tokens"].sort_values(ascending=False)
    if tok_vals.sum() > 0:
        tok_vals.plot(kind="bar", ax=axes[1], color="seagreen")
        axes[1].set_title("Avg tokens per agent node")
        axes[1].set_xlabel("")
        axes[1].set_ylabel("Tokens")
        axes[1].tick_params(axis="x", rotation=20)
    else:
        axes[1].text(0.5, 0.5, "No token data", ha="center", va="center",
                     transform=axes[1].transAxes)

    plt.tight_layout()
    plt.show()
    plt.close("all")

## 6 — LangGraph routing path frequency

Reconstructed from `trace_id` grouping — ordered agent sequence per query.

In [ ]:
if not agent_df.empty and not runs_df.empty:
    paths = (
        agent_df.dropna(subset=["trace_id"])
        .sort_values("start_time")
        .groupby("trace_id")["name"]
        .apply(lambda names: " → ".join(names))
        .value_counts()
        .reset_index()
    )
    paths.columns = ["path", "count"]
    paths["pct"] = (paths["count"] / paths["count"].sum() * 100).round(1)

    print(f"{len(paths)} distinct routing paths")
    display(paths.head(15))

    if 1 < len(paths) <= 12:
        fig, ax = plt.subplots(figsize=(10, max(3, len(paths) * 0.5)))
        paths.set_index("path")["count"].sort_values().plot(
            kind="barh", ax=ax, color="steelblue")
        ax.set_title("LangGraph routing path frequency")
        ax.set_xlabel("Count")
        plt.tight_layout()
        plt.show()
        plt.close("all")

## 7 — Latency and token usage over time

In [ ]:
if not runs_df.empty and len(runs_df) >= 2:
    ts = runs_df.set_index("start_time").sort_index()
    # Use a frequency that makes sense for the data span
    span_h = (ts.index[-1] - ts.index[0]).total_seconds() / 3600 if len(ts) > 1 else 1
    freq = "10min" if span_h < 2 else ("30min" if len(runs_df) >= 10 else "1h")
    hourly_lat = ts["duration_s"].resample(freq).agg(
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95),
    ).dropna()
    hourly_tok = ts[["prompt_tokens", "completion_tokens"]].resample(freq).sum()

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    if len(hourly_lat) >= 2:
        axes[0].plot(hourly_lat.index, hourly_lat["p50"], label="p50", color="steelblue")
        axes[0].fill_between(hourly_lat.index, hourly_lat["p50"], hourly_lat["p95"],
                             alpha=0.2, color="steelblue", label="p50–p95")
        axes[0].plot(hourly_lat.index, hourly_lat["p95"], label="p95",
                     color="orange", linestyle="--")
    else:
        # Not enough buckets for a line — scatter instead
        axes[0].scatter(runs_df["start_time"], runs_df["duration_s"],
                        color="steelblue", s=40)
    axes[0].set_title("Latency over time")
    axes[0].set_xlabel("UTC")
    axes[0].set_ylabel("Duration (s)")
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %Hh"))
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30)
    axes[0].legend(fontsize=8)

    if hourly_tok.sum().sum() > 0:
        hourly_tok.plot(kind="area", stacked=True, ax=axes[1], alpha=0.75,
                        color=["#aec7e8", "#1f77b4"])
        axes[1].set_title("Token usage per bucket")
        axes[1].set_xlabel("UTC")
        axes[1].set_ylabel("Tokens")
        axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %Hh"))
        plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)
    else:
        axes[1].scatter(runs_df["start_time"], runs_df["duration_s"],
                        alpha=0.6, s=30, color="seagreen")
        axes[1].set_title("Latency scatter (no token data)")
        axes[1].set_xlabel("UTC")
        axes[1].set_ylabel("Duration (s)")
        axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %Hh"))
        plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)

    plt.tight_layout()
    plt.show()
    plt.close("all")

## 8 — Error runs

In [11]:
error_runs = runs_df[runs_df["error"].notna()].copy()
print(f"Error runs: {len(error_runs)} of {len(runs_df)}")
if not error_runs.empty:
    pd.set_option("display.max_colwidth", 300)
    display(
        error_runs[["start_time", "run_id", "duration_s", "error"]]
        .reset_index(drop=True)
    )

Error runs: 0 of 12


## 9 — Full trace replay

Every node in execution order with type, status, duration, and token count.

In [12]:
# Leave as None to auto-pick the slowest run
TRACE_RUN_ID = None

if TRACE_RUN_ID is None and not runs_df.empty:
    TRACE_RUN_ID = runs_df.loc[runs_df["duration_s"].idxmax(), "run_id"]
    print(f"Auto-selected slowest run: {TRACE_RUN_ID}")

if TRACE_RUN_ID:
    root = client.read_run(TRACE_RUN_ID)
    trace_id = str(root.trace_id) if root.trace_id else TRACE_RUN_ID

    children = sorted(
        [r for r in client.list_runs(
            project_name=PROJECT_NAME,
            trace_id=trace_id,
        ) if str(r.id) != TRACE_RUN_ID],
        key=lambda r: r.start_time or datetime.min.replace(tzinfo=timezone.utc)
    )

    dur = (
        f"{(root.end_time - root.start_time).total_seconds():.2f}s"
        if root.end_time and root.start_time else "?"
    )
    print(f"Trace : {TRACE_RUN_ID}")
    print(f"Status: {root.status}  |  Duration: {dur}  |  Tokens: {root.total_tokens}")
    if root.error:
        print(f"Error : {root.error[:300]}")

    print(f"\n{'Node':<30} {'Type':<10} {'Status':<10} {'Duration':>10}  {'Tokens':>8}")
    print("-" * 76)
    for r in children:
        node_dur = (
            f"{(r.end_time - r.start_time).total_seconds():.3f}s"
            if r.end_time and r.start_time else "—"
        )
        print(f"{r.name[:29]:<30} {r.run_type:<10} {r.status:<10} "
              f"{node_dur:>10}  {r.total_tokens or 0:>8}")

    print("\n── Root inputs ──")
    print(json.dumps(root.inputs, indent=2, default=str)[:1500])
    print("\n── Root outputs ──")
    print(json.dumps(root.outputs, indent=2, default=str)[:1500])

Auto-selected slowest run: 6f802895-f8d7-47b2-aeff-92db4d36ab99
Trace : 6f802895-f8d7-47b2-aeff-92db4d36ab99
Status: success  |  Duration: 3.11s  |  Tokens: 1817

Node                           Type       Status       Duration    Tokens
----------------------------------------------------------------------------
__start__                      chain      success        0.000s         0
assistant_agent                chain      success        0.573s       788
ChatGroq                       llm        success        0.569s       788
ChannelWrite<assistant_agent,  chain      success        0.000s         0
assitant_router                chain      success        0.000s         0
GoogleMaps                     chain      success        1.597s         0
ChannelWrite<GoogleMaps,messa  chain      success        0.001s         0
googlemaps_router              chain      success        0.000s         0
generator_agent                chain      success        0.886s      1029
ChatGroq            

## 10 — LLM node drill-down

Exact messages sent to the model and the raw reply.

In [13]:
# Leave as None to auto-pick the first LLM child from the trace above
NODE_RUN_ID = None

if NODE_RUN_ID is None and TRACE_RUN_ID:
    llm_children = [r for r in children if r.run_type == "llm"]
    if llm_children:
        NODE_RUN_ID = str(llm_children[0].id)
        print(f"Auto-selected first LLM node: {llm_children[0].name}")
    else:
        print("No LLM-type children in this trace.")

if NODE_RUN_ID:
    node = client.read_run(NODE_RUN_ID)
    dur = (
        f"{(node.end_time - node.start_time).total_seconds():.2f}s"
        if node.end_time and node.start_time else "?"
    )
    print(f"Node  : {node.name}")
    print(f"Status: {node.status}  |  Duration: {dur}")
    print(f"Tokens: prompt={node.prompt_tokens}  completion={node.completion_tokens}")

    print("\n── Messages sent to LLM ──")
    messages = (
        node.inputs.get("messages")
        or node.inputs.get("prompts")
        or []
    )
    for msg in messages:
        if isinstance(msg, dict):
            role    = msg.get("role") or msg.get("type") or "?"
            content = str(msg.get("content") or "")[:600]
        else:
            role, content = "?", str(msg)[:600]
        print(f"  [{role}]  {content}\n")

    print("── LLM reply ──")
    print(json.dumps(node.outputs, indent=2, default=str)[:2000])

Auto-selected first LLM node: ChatGroq
Node  : ChatGroq
Status: success  |  Duration: 0.57s
Tokens: prompt=667  completion=121

── Messages sent to LLM ──
  [?]  [{'id': ['langchain', 'schema', 'messages', 'SystemMessage'], 'kwargs': {'content': '\nAct as Localelive (an assistant integrated to IoT search engine)\nAnswer user queries ONLY if you can\nUser queries may fall into one of the following categories:\n1- Greeting/General: Greeting you (e.g., hello, hi, hey, thanks, thank you, bye) or answering a general question that you can ONLY answer, or thank you at the end of the conversations.\n2- Service Recommendation: Asking for a recommendation for a service or a place to visit, OR mentioning a specific brand, restaurant, store, or business name. Exam

── LLM reply ──
{
  "generations": [
    [
      {
        "generation_info": {
          "finish_reason": "stop",
          "logprobs": null
        },
        "message": {
          "id": [
            "langchain",
            "schema

## 11 — Export to CSV

In [14]:
if not runs_df.empty:
    out = Path("localelive_langsmith_runs.csv")
    runs_df.drop(columns=["error"], errors="ignore").to_csv(out, index=False)
    print(f"Exported {len(runs_df)} root runs → {out}")

    summary = {
        "project":            PROJECT_NAME,
        "window_hours":       LOOKBACK_HOURS,
        "total_runs":         len(runs_df),
        "error_rate":         round(runs_df["error"].notna().mean(), 4),
        "p50_latency_s":      round(runs_df["duration_s"].quantile(0.5), 3),
        "p95_latency_s":      round(runs_df["duration_s"].quantile(0.95), 3),
        "total_tokens":       int(runs_df["total_tokens"].sum()),
        "avg_tokens_per_run": round(runs_df["total_tokens"].mean(), 1),
        "total_cost_usd":     round(runs_df["total_cost"].sum(), 6),
    }
    print(json.dumps(summary, indent=2))

Exported 12 root runs → localelive_langsmith_runs.csv
{
  "project": "default",
  "window_hours": 72,
  "total_runs": 12,
  "error_rate": 0.0,
  "p50_latency_s": 2.607,
  "p95_latency_s": 3.093,
  "total_tokens": 22933,
  "avg_tokens_per_run": 1911.1,
  "total_cost_usd": 0.0
}
